# Análisis de Calidad de Datos: Customer Journey - Paigo
**Autor:** Franco Robotti  
**Objetivo:** Auditar la calidad de las cuatro tablas del journey (`clientes`, `productos`, `reviews_stores`, `bot_chats`) antes del análisis: integridad referencial, consistencia cronológica, dominios, duplicados, nulos y coherencia de negocio.  definición de una **fuente de verdad** y exportación de un dataset limpio.

> El análisis exploratorio se desarrolla en `00_analisis_exploratorio`; las consultas analíticas en `02_sql_analitico`.

## 1. Importación y carga de datos
Cargamos las cuatro hojas y normalizamos las fechas. Usamos **DuckDB** para consultar los DataFrames directamente con SQL.

In [1]:
import os
import duckdb
import numpy as np
import pandas as pd

DATA_PATH = "../data/raw"
df_clientes  = pd.read_excel(f"{DATA_PATH}/clientes.xlsx")
df_productos = pd.read_excel(f"{DATA_PATH}/productos.xlsx")
df_reviews   = pd.read_excel(f"{DATA_PATH}/reviews_stores.xlsx")
df_bot       = pd.read_excel(f"{DATA_PATH}/bot_chats.xlsx")

for col in ["fecha_alta"]:
    df_clientes[col] = pd.to_datetime(df_clientes[col], errors="coerce")
df_reviews["fecha_review"] = pd.to_datetime(df_reviews["fecha_review"], errors="coerce")
df_bot["fecha_inicio"]     = pd.to_datetime(df_bot["fecha_inicio"], errors="coerce")

for nombre, df in [("clientes", df_clientes), ("productos", df_productos),
                   ("reviews_stores", df_reviews), ("bot_chats", df_bot)]:
    print(f"{nombre:15s} -> {df.shape[0]:>5} filas x {df.shape[1]:>2} columnas")

clientes        ->  1500 filas x  7 columnas
productos       ->     5 filas x  8 columnas
reviews_stores  ->  2200 filas x 10 columnas
bot_chats       ->  3000 filas x 11 columnas


## 2. Auditoría y calidad de datos

### 2.1. Integridad referencial
Verificamos que las claves foráneas (`id_producto`, `id_cliente`) tengan correspondencia en sus tablas maestras.

In [3]:
reviews_sin_producto = duckdb.query('''
SELECT
    r.id_producto,
    COUNT(*) AS cantidad_reviews
FROM df_reviews r
LEFT JOIN df_productos p ON r.id_producto = p.id_producto
WHERE p.id_producto IS NULL
GROUP BY r.id_producto
ORDER BY cantidad_reviews DESC
''').df()

huerfanas = int(reviews_sin_producto["cantidad_reviews"].sum())
integridad = (len(df_reviews) - huerfanas) / len(df_reviews) * 100
print(f"Reviews con id_producto inexistente: {huerfanas} ({100-integridad:.1f}%)")
print(f"Integridad referencial reviews->productos: {integridad:.1f}%")
print("Productos del catálogo:", sorted(df_productos['id_producto'].tolist()))
display(reviews_sin_producto)

Reviews con id_producto inexistente: 805 (36.6%)
Integridad referencial reviews->productos: 63.4%
Productos del catálogo: [1, 2, 3, 4, 5]


,id_producto,cantidad_reviews
0,6,296
1,8,263
2,7,246


In [4]:
chequeos = {
    "reviews -> clientes": "SELECT COUNT(*) n FROM df_reviews r LEFT JOIN df_clientes c ON r.id_cliente=c.id_cliente WHERE c.id_cliente IS NULL",
    "bot -> clientes":     "SELECT COUNT(*) n FROM df_bot b LEFT JOIN df_clientes c ON b.id_cliente=c.id_cliente WHERE c.id_cliente IS NULL",
    "bot -> productos":    "SELECT COUNT(*) n FROM df_bot b LEFT JOIN df_productos p ON b.id_producto=p.id_producto WHERE p.id_producto IS NULL",
}
for nombre, q in chequeos.items():
    n = duckdb.query(q).df()["n"][0]
    print(f"{nombre:20s} -> huerfanos: {n}")

reviews -> clientes  -> huerfanos: 0
bot -> clientes      -> huerfanos: 0
bot -> productos     -> huerfanos: 0


**Interpretación y hallazgos:**
* **805 reviews (36.6%)** apuntan a `id_producto` **6, 7 y 8**, que **no existen** en el catálogo (`productos` solo tiene IDs 1–5). La integridad reviews→productos es de **63.4%**.
* El resto de las relaciones está **íntegro**: reviews→clientes, bot→clientes y bot→productos no tienen huérfanos.
* **Causa probable:** catálogo de productos incompleto (faltan 3 productos), no reviews inválidas. Decisión en la sección 2.7: **conservar** esas reviews etiquetando el producto como "Desconocido", no descartarlas.

### 2.2. Consistencia cronológica
Una interacción (review o chat) no puede ocurrir **antes** de que el cliente exista (`fecha_alta`), ni en el futuro.

In [8]:
reviews_antes_alta = duckdb.query('''
SELECT COUNT(*) AS reviews_antes_del_alta
FROM df_reviews r
JOIN df_clientes c ON r.id_cliente = c.id_cliente
WHERE r.fecha_review < c.fecha_alta
''').df()

chats_antes_alta = duckdb.query('''
SELECT COUNT(*) AS chats_antes_del_alta
FROM df_bot b
JOIN df_clientes c ON b.id_cliente = c.id_cliente
WHERE b.fecha_inicio < c.fecha_alta
''').df()

print("Reviews anteriores al alta del cliente:",
      int(reviews_antes_alta.iloc[0,0]),
      f"({reviews_antes_alta.iloc[0,0]/len(df_reviews)*100:.1f}%)")
print("Chats anteriores al alta del cliente  :",
      int(chats_antes_alta.iloc[0,0]),
      f"({chats_antes_alta.iloc[0,0]/len(df_bot)*100:.1f}%)")

hoy = pd.Timestamp("2026-06-19")
print("\nReviews con fecha futura:", int((df_reviews['fecha_review'] > hoy).sum()))
print("Chats con fecha futura  :", int((df_bot['fecha_inicio'] > hoy).sum()))
print("\nRango fechas alta   :", df_clientes['fecha_alta'].min().date(), "->", df_clientes['fecha_alta'].max().date())
print("Rango fechas review :", df_reviews['fecha_review'].min().date(), "->", df_reviews['fecha_review'].max().date())

Reviews anteriores al alta del cliente: 1115 (50.7%)
Chats anteriores al alta del cliente  : 1527 (50.9%)

Reviews con fecha futura: 0
Chats con fecha futura  : 0

Rango fechas alta   : 2023-01-02 -> 2025-03-30
Rango fechas review : 2023-01-01 -> 2025-03-30


**Interpretación y hallazgos:**
* **1.115 reviews (50.7%)** y **1.527 chats (50.9%)** tienen fecha **anterior al alta** del cliente. Es cronológicamente imposible: un cliente no puede interactuar antes de existir.
* No hay fechas futuras, así que el problema no es de corte sino de **coherencia entre `fecha_alta` y las fechas de evento** (probablemente generadas de forma independiente).
* **Impacto:** bloquea cualquier métrica de **ciclo de vida relativa** (antigüedad al momento de la review, días desde el alta hasta el primer chat). Se documenta como **limitación** y se marca con un flag; no se debe calcular "tiempo desde el alta" hasta clarificar con el equipo de datos.

### 2.3. Validación de dominios y rangos
Verificamos que las métricas estén dentro de sus rangos válidos según el diccionario.

In [10]:
validaciones = duckdb.query('''
SELECT 'rating fuera de 1-5'      AS chequeo, COUNT(*) AS violaciones FROM df_reviews WHERE rating < 1 OR rating > 5
UNION ALL SELECT 'csat fuera de 1-5',  COUNT(*) FROM df_bot WHERE csat_score IS NOT NULL AND (csat_score < 1 OR csat_score > 5)
UNION ALL SELECT 'nps fuera de 0-10',  COUNT(*) FROM df_clientes WHERE nps_score IS NOT NULL AND (nps_score < 0 OR nps_score > 10)
UNION ALL SELECT 'duracion <= 0',      COUNT(*) FROM df_bot WHERE duracion_segundos <= 0
UNION ALL SELECT 'mensajes <= 0',      COUNT(*) FROM df_bot WHERE mensajes_enviados <= 0
''').df()
display(validaciones)

,chequeo,violaciones
0,rating fuera de 1-5,0
1,csat fuera de 1-5,0
2,nps fuera de 0-10,0
3,duracion <= 0,0
4,mensajes <= 0,0


**Interpretación y hallazgos:**
* **Sin violaciones de dominio:** `rating` (1–5), `csat` (1–5) y `nps` (0–10) están en rango; no hay duraciones ni cantidades de mensajes ≤ 0.

### 2.4. Auditoría de duplicados
Buscamos duplicados por clave primaria y duplicados *lógicos* (misma interacción cargada dos veces con distinto ID).

In [11]:
dups = {}
for nombre, dfname, pk in [("reviews_stores","df_reviews","id_review"),
                           ("bot_chats","df_bot","id_chat"),
                           ("clientes","df_clientes","id_cliente"),
                           ("productos","df_productos","id_producto")]:
    q = f"SELECT {pk}, COUNT(*) c FROM {dfname} GROUP BY {pk} HAVING COUNT(*) > 1"
    dups[nombre] = len(duckdb.query(q).df())
    print(f"{nombre:15s} | PK {pk:12s} duplicada: {dups[nombre]}")

# Duplicado lógico de reviews (misma persona, producto, fecha y rating)
soft = duckdb.query('''
SELECT id_cliente, id_producto, fecha_review, rating, COUNT(*) c
FROM df_reviews
GROUP BY 1,2,3,4
HAVING COUNT(*) > 1
''').df()
print("\nDuplicados lógicos de reviews (cliente+producto+fecha+rating):", len(soft))

reviews_stores  | PK id_review    duplicada: 0
bot_chats       | PK id_chat      duplicada: 0
clientes        | PK id_cliente   duplicada: 0
productos       | PK id_producto  duplicada: 0

Duplicados lógicos de reviews (cliente+producto+fecha+rating): 0


**Interpretación y hallazgos:**
* **No hay duplicados** por clave primaria en ninguna tabla, ni duplicados lógicos en reviews.
* *Nota:* que un cliente tenga varias reviews o chats **no es un duplicado** sino comportamiento legítimo (ver granularidad en el notebook 00). No se aplica deduplicación.

### 2.5. Análisis de valores nulos
Auditoría de nulos por tabla y análisis del patrón en las métricas clave.

In [16]:
for nombre, df in [("clientes", df_clientes), ("productos", df_productos),
                   ("reviews_stores", df_reviews), ("bot_chats", df_bot)]:
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f"== {nombre} ==")
    if len(nulos):
        for col, n in nulos.items():
            print(f"{col:22s}: {n:>4} ({n/len(df)*100:.1f}%)")
    else:
        print("sin nulos")
    print()

== clientes ==
nps_score             :  121 (8.1%)

== productos ==
sin nulos

== reviews_stores ==
sin nulos

== bot_chats ==
duracion_segundos     :  140 (4.7%)
csat_score            : 1653 (55.1%)



In [17]:
# Patrón de nulos en csat por resolución y por escalado
print("CSAT nulls por resolución:")
display(duckdb.query('''
SELECT resolucion,
       COUNT(*) AS total,
       SUM(CASE WHEN csat_score IS NULL THEN 1 ELSE 0 END) AS nulls,
       ROUND(100.0*SUM(CASE WHEN csat_score IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_null
FROM df_bot GROUP BY resolucion ORDER BY pct_null DESC
''').df())

print("NPS nulls por segmento:")
display(duckdb.query('''
SELECT segmento,
       COUNT(*) AS total,
       SUM(CASE WHEN nps_score IS NULL THEN 1 ELSE 0 END) AS nulls,
       ROUND(100.0*SUM(CASE WHEN nps_score IS NULL THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_null
FROM df_clientes GROUP BY segmento ORDER BY pct_null DESC
''').df())

CSAT nulls por resolución:


,resolucion,total,nulls,pct_null
0,Abandono,461,264.0,57.3
1,Sin resolución,318,180.0,56.6
2,Resuelto bot,1470,804.0,54.7
3,Derivado humano,751,405.0,53.9


NPS nulls por segmento:


,segmento,total,nulls,pct_null
0,Premium,214,19.0,8.9
1,Nuevo,459,37.0,8.1
2,Recurrente,581,47.0,8.1
3,Inactivo,246,18.0,7.3


**Interpretación y hallazgos:**
* **`csat_score`: 1.653 nulos (55.1%)** más de la mitad de los chats no tienen puntaje. Es esperable: el CSAT depende de que el usuario responda la encuesta post-chat.
* **`nps_score`: 121 nulos (8.1%)** y **`duracion_segundos`: 140 nulos (4.7%)**. La duración nula no estaba en la auditoría previa y debe contemplarse al promediar tiempos.
* **Patrón:** los nulos de CSAT son **parejos por resolución** (54–57%) y los de NPS **parejos por segmento** (7–9%). No hay un patrón fuerte → se comportan como **MCAR** (aleatorios).
* **Decisión:** **no imputar**. Imputar CSAT/NPS introduciría sesgo; las métricas se calculan sobre los registros no nulos, reportando la cobertura.

### 2.6. Validación de consistencia interna

In [19]:
coherencia = duckdb.query('''
SELECT
    resolucion,
    escalado_humano,
    COUNT(*) AS cantidad
FROM df_bot
GROUP BY resolucion, escalado_humano
ORDER BY resolucion, escalado_humano
''').df()
display(coherencia)

contradiccion = duckdb.query('''
SELECT
    SUM(CASE WHEN resolucion='Derivado humano' AND escalado_humano=0 THEN 1 ELSE 0 END) AS derivado_pero_no_escalado,
    SUM(CASE WHEN resolucion<>'Derivado humano' AND escalado_humano=1 THEN 1 ELSE 0 END) AS escalado_pero_no_derivado
FROM df_bot
''').df()
display(contradiccion)

,resolucion,escalado_humano,cantidad
0,Abandono,False,346
1,Abandono,True,115
2,Derivado humano,False,562
3,Derivado humano,True,189
4,Resuelto bot,False,1107
5,Resuelto bot,True,363
6,Sin resolución,False,211
7,Sin resolución,True,107


,derivado_pero_no_escalado,escalado_pero_no_derivado
0,562.0,585.0


**Interpretación y hallazgos:**
* **562 de 751** chats con `resolucion = 'Derivado humano'` tienen `escalado_humano = 0` (75%), y otros **585** tienen `escalado_humano = 1` sin estar derivados. Los dos campos **no son coherentes** entre sí.
* **Impacto:** no se pueden usar ambos como sinónimos. Para medir "intervención humana" hay que **elegir una fuente de verdad**. Se recomienda priorizar `resolucion` (describe el desenlace) y tratar `escalado_humano` con cautela, dejándolo documentado para el equipo de datos.

### 2.7. Fuente de verdad: dataset limpio y decisiones
Aplicamos las decisiones de la auditoría y exportamos tablas limpias y trazables (con flags, sin destruir información).

In [20]:
# --- REVIEWS: etiquetar producto desconocido y flag de cronología ---
df_reviews_clean = df_reviews.copy()
ids_validos = set(df_productos["id_producto"])
df_reviews_clean["producto_valido"] = df_reviews_clean["id_producto"].isin(ids_validos)

df_reviews_clean = df_reviews_clean.merge(
    df_clientes[["id_cliente", "fecha_alta"]], on="id_cliente", how="left")
df_reviews_clean["fecha_inconsistente"] = df_reviews_clean["fecha_review"] < df_reviews_clean["fecha_alta"]

# --- BOT: flag de cronología (no se imputa csat ni duracion) ---
df_bot_clean = df_bot.merge(
    df_clientes[["id_cliente", "fecha_alta"]], on="id_cliente", how="left")
df_bot_clean["fecha_inconsistente"] = df_bot_clean["fecha_inicio"] < df_bot_clean["fecha_alta"]

print("Reviews marcadas con producto desconocido:", int((~df_reviews_clean['producto_valido']).sum()))
print("Reviews con fecha inconsistente          :", int(df_reviews_clean['fecha_inconsistente'].sum()))
print("Chats con fecha inconsistente            :", int(df_bot_clean['fecha_inconsistente'].sum()))

Reviews marcadas con producto desconocido: 805
Reviews con fecha inconsistente          : 1115
Chats con fecha inconsistente            : 1527


In [ ]:
os.makedirs("../data/clean", exist_ok=True)
df_clientes.to_excel("../data/clean/clientes_limpio.xlsx", index=False)
df_productos.to_excel("../data/clean/productos_limpio.xlsx", index=False)
df_reviews_clean.to_excel("../data/clean/reviews_limpio.xlsx", index=False)
df_bot_clean.to_excel("../data/clean/bot_chats_limpio.xlsx", index=False)
print("Datasets limpios exportados a ../data/clean/")

Datasets limpios exportados a ../data/clean/


## 3. Síntesis de la auditoría de calidad

| Dimensión | Resultado | Decisión |
|:---|:---|:---|
| Integridad reviews→productos | 805 reviews (36.6%) con producto inexistente (id 6,7,8) | Conservar + flag `producto_valido` |
| Integridad clientes (reviews/bot) | Sin huérfanos | — |
| Consistencia cronológica | ~51% de reviews y chats anteriores al alta | Flag `fecha_inconsistente`; limita métricas de ciclo de vida |
| Dominios y rangos | rating/csat/nps en rango; sin duraciones negativas | — |
| Duplicados | Ninguno (PK ni lógico) | Sin deduplicación |
| Nulos | csat 55.1%, nps 8.1%, **duración 4.7%**; patrón ≈ MCAR | No imputar; calcular sobre no nulos |
| Coherencia escalado vs. resolución | 562/751 derivados sin flag de escalado | Elegir `resolucion` como fuente de verdad |

**Conclusión:** los datos son utilizables para análisis agregado de satisfacción y comportamiento, pero con **dos limitaciones a comunicar**: el catálogo de productos incompleto y la inconsistencia cronológica que impide métricas de antigüedad relativa. El dataset limpio queda en `../data/clean/` con flags de trazabilidad.